In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Yeast


In [3]:
# sgd/ALL_YEAST_GENE_PHENOTYPE.csv

# Databases Having Gene PHENOTYPE relation of yeast 

In [4]:
#

In [5]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Yeast_gene_phenotype
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Yeast/Yeast_gene_phenotype/Yeast_gene_phenotype.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name'
]

mkdir: cannot create directory ‘Yeast_gene_phenotype’: File exists


In [6]:
sgd = pd.read_csv(
    f'{BASE_DIR}data_collection/databases_for_mapping/sgd/SGD_features.tab',
    sep='\t',
    header=None,
    usecols=[3, 4, 5],          # systematic_name, gene_name, description
    names=['Systematic_name', 'Gene_name', 'Description'],
    dtype=str
)
sgd = sgd[~sgd['Description'].isna()]
# Drop rows without a systematic name (YIL064W-style)
# sgd = sgd[sgd['Systematic_name'].str.match(r'^Y[A-Z]{2}\d+[CW]', na=False)]
sgd
yeast_sys2name_dict = dict(zip(sgd['Gene_name'], sgd['Systematic_name']))
# yeast_sys2name_dict

In [7]:
sgd_gene_phenotype = pd.read_csv(PROC_DIR + 'sgd/ALL_YEAST_GENE_PHENOTYPE.csv')
sgd_gene_phenotype.columns = sgd_gene_phenotype.columns.str.lower()
sgd_gene_phenotype['head_id_is'] = 'SGD_SysematicName'
sgd_gene_phenotype['species'] = 'S.cerevisae'
sgd_gene_phenotype['kg_type'] = 'Generalised'
sgd_gene_phenotype['relation_type'] = ''

print(f"sgd_gene_phenotype: {len(sgd_gene_phenotype):,} rows")
sgd_gene_phenotype.head(3)


sgd_gene_phenotype: 73,397 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_type
0,Q0045,Gene_Phenotype,hydrostatic pressure resistance: decreased,Gene,,Phenotype,SGD,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_hydrostatic pressure resist...,S.cerevisae,Generalised
1,Q0045,Gene_Phenotype,petite,Gene,,Phenotype,SGD,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_petite,S.cerevisae,Generalised
2,Q0045,Gene_Phenotype,respiratory growth: absent,Gene,,Phenotype,SGD,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_respiratory growth: absent,S.cerevisae,Generalised


In [8]:
# List all source DataFrames to include
source_dfs = [
    sgd_gene_phenotype,
    # ← add further source DataFrames here
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 73,397


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,Q0045,Gene_Phenotype,hydrostatic pressure resistance: decreased,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_hydrostatic pressure resist...
1,Q0045,Gene_Phenotype,petite,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_petite
2,Q0045,Gene_Phenotype,respiratory growth: absent,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_respiratory growth: absent
3,Q0045,Gene_Phenotype,viable,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_viable
4,Q0080,Gene_Phenotype,respiratory growth: absent,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,F1F0 ATP synthase subunit 8,classical genetics_respiratory growth: absent
...,...,...,...,...,...,...,...,...,...,...,...,...
73392,YPR201W,Gene_Phenotype,stress resistance: decreased,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,Arr3p,"homozygous diploid, competitive growth_stress ..."
73393,YPR201W,Gene_Phenotype,utilization of nitrogen source: decreased rate,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,Arr3p,systematic mutation set (prototrophic version ...
73394,YPR201W,Gene_Phenotype,vacuolar morphology: abnormal,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,Arr3p,systematic mutation set (screen all non-essent...
73395,YPR201W,Gene_Phenotype,vegetative growth: decreased rate,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,Arr3p,large-scale survey_vegetative growth: decrease...


# Sanity Check — Distinct Values

In [9]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Gene_Phenotype'}
head_type           : {'Gene'}
tail_type           : {'Phenotype'}
relation_type       : {''}
kg_source           : {'SGD'}
head_id_is          : {'SGD_SysematicName'}
tail_id_is          : {nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan

In [10]:

# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 73,397 remaining


In [11]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,73397,73397,146794


# Deduplication

In [12]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 73,397  |  After dedup: 73,397


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,Q0045,Gene_Phenotype,hydrostatic pressure resistance: decreased,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_hydrostatic pressure resist...
1,Q0045,Gene_Phenotype,petite,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_petite
2,Q0045,Gene_Phenotype,respiratory growth: absent,Gene,,Phenotype,SGD,Generalised,SGD_SysematicName,NaN,cytochrome c oxidase subunit 1,classical genetics_respiratory growth: absent


In [13]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,73397,73397,146794


In [14]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'SGD'} kg_source
SGD    73397
Name: count, dtype: int64


In [15]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    73397
Name: count, dtype: int64


In [16]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 73,397 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Yeast/Yeast_gene_phenotype/Yeast_gene_phenotype.csv
